# Solar Filament Segmentation Challenge 2026 — reproduce the final submission from the stored weights

This notebook reproduces the final predictions **without any training**: it loads the stored weights (`weights/final_model.pth`), runs inference on the competition
test images and writes `submission.csv` in the competition format (`filament_id`, `segmentation_rle`).

## The pipeline in one view
1. **Input.** Each 2048x2048 grayscale solar H-alpha image is resized to **1024x1024** (area interpolation) and normalised with its own 1st-99th percentile to [0, 1].
2. **Model.** A **U-Net with an ImageNet-pretrained ResNet-50 encoder** (torchvision) and a 5-stage U-Net decoder (bilinear up-sampling + two 3x3 conv-BN-ReLU blocks per stage,
   skip connections from the encoder stages). The grayscale image is repeated to 3 channels and ImageNet-normalised inside the model. Output: one logit map.
3. **Test-time augmentation.** Predictions for the image, its horizontal flip, vertical flip and both flips are averaged (4 views).
4. **Post-processing (instance extraction).** Probability map up-sampled to 2048x2048 (bilinear) -> threshold 0.5 -> morphological **closing** (elliptical kernel, radius 10 px) to re-join
   fragments -> **8-connected components** = filament instances -> discard instances of **<= 400 px**.
5. **Output.** Each instance is written as a COCO run-length encoding (`pycocotools`, Fortran order, 2048x2048) in `submission.csv`; `filament_id = <image_id>_<k>`.

## How the weights were obtained (training is NOT needed to run this notebook)
`notebooks/05_train_final_model.ipynb` (also in this repository) trains the model on all 707 labelled images: BCE + soft-Dice loss, Adam (decoder lr 1e-3, encoder lr 1e-4, weight decay 1e-5) with cosine decay,
batch 4, 50 epochs, mixed precision, flip augmentation, a random annotator record per image per epoch, and re-estimation of the BatchNorm statistics at the end (already contained in the stored weights).
The stored file keeps the weights in half precision (to stay below GitHub's file-size limit); they are converted back to float32 when loaded.

**Requirements:** see `requirements.txt` (the cell below also prints the installed versions). A GPU makes inference take a few minutes; on CPU it takes much longer.

In [ ]:
# On Kaggle (Internet ON) the repository can be fetched so that weights/final_model.pth is available; uncomment the next line:
# !git clone https://github.com/parsaBehjatTabrizi/solar-filament-segmentation-2026

In [ ]:
import os, sys, platform, importlib.metadata as md_
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from pycocotools import mask as mask_util

PKGS = ["torch", "torchvision", "numpy", "pandas", "opencv-python-headless", "pycocotools", "Pillow", "tqdm", "scipy"]
vers = {}
for p in PKGS:
    try:
        vers[p] = md_.version(p)
    except Exception:
        alt = {"opencv-python-headless": "opencv-python"}.get(p)
        try:
            vers[p] = md_.version(alt) if alt else "not installed"
        except Exception:
            vers[p] = "not installed"
print("python", sys.version.split()[0], "|", platform.platform())
for k, v in vers.items():
    print(f"  {k}=={v}")
Path("requirements_auto.txt").write_text("\n".join(f"{k}=={v}" for k, v in vers.items() if v != "not installed") + "\n")
print("wrote requirements_auto.txt (copy the versions into requirements.txt)")

In [ ]:
# ============================================================
# CONFIGURATION — paths are searched automatically; edit only if the search fails
# ============================================================
RESOLUTION = 1024
THRESHOLD, CLOSING_RADIUS, MIN_PIXELS = 0.5, 10, 400
FLIP_TTA = True

def first_existing(cands):
    for c in cands:
        if c and Path(c).exists():
            return Path(c)
    return None

def find_weights():
    local = first_existing(["weights/final_model.pth", "../weights/final_model.pth",
                            "weights/final_weights_fp16.pth", "../weights/final_weights_fp16.pth"])
    if local is not None:
        return local
    if Path("/kaggle/input").exists():                       # e.g. when the weights were attached as a Kaggle input
        for name in ("final_model.pth", "final_weights_fp16.pth"):
            hit = next(iter(Path("/kaggle/input").rglob(name)), None)
            if hit is not None:
                return hit
    # e.g. when this repository was cloned into the working directory (git clone ...)
    for name in ("final_model.pth", "final_weights_fp16.pth"):
        hit = next(iter(Path(".").rglob(name)), None)
        if hit is not None:
            return hit
    return None

WEIGHTS_PATH = find_weights()

def find_test_dir():
    explicit = os.environ.get("TEST_IMAGES_DIR")
    if explicit and Path(explicit).exists():
        return Path(explicit)
    for root in (Path("/kaggle/input"), Path("."), Path("..")):
        if root.exists():
            for d in sorted(root.rglob("test_images")):
                if d.is_dir():
                    return d
    return None

TEST_DIR = find_test_dir()
def find_reference():
    """Optional: the CSV that was actually submitted, used only for the agreement check at the end."""
    local = first_existing(["submission_F01_resnet50_s0_full.csv", "reference_submission.csv", "../reference_submission.csv"])
    if local is not None:
        return local
    if Path("/kaggle/input").exists():
        return next(iter(Path("/kaggle/input").rglob("submission_F01_resnet50_s0_full.csv")), None)
    return None

REFERENCE_CSV = find_reference()
OUT_CSV = Path("submission.csv")

print("weights:", WEIGHTS_PATH)
print("test images dir:", TEST_DIR)
print("reference csv (optional):", REFERENCE_CSV)
assert WEIGHTS_PATH is not None, "weights file not found: put final_model.pth in ./weights/"
assert TEST_DIR is not None, "test_images directory not found: set the environment variable TEST_IMAGES_DIR"

In [ ]:
# ============================================================
# MODEL: U-Net with a ResNet-50 encoder
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class ResNetUNet(nn.Module):
    """ResNet-50 encoder (stages: /2:64, /4:256, /8:512, /16:1024, /32:2048 channels) + U-Net decoder."""
    def __init__(self):
        super().__init__()
        r = torchvision.models.resnet50(weights=None)          # weights come from the stored file, no download needed
        c = [64, 256, 512, 1024, 2048]
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.d4 = ConvBlock(c[4] + c[3], 256)
        self.d3 = ConvBlock(256 + c[2], 128)
        self.d2 = ConvBlock(128 + c[1], 64)
        self.d1 = ConvBlock(64 + c[0], 64)
        self.d0 = ConvBlock(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):                                       # x: (B,1,H,W) in [0,1]
        x3 = (x.repeat(1, 3, 1, 1) - self.mean) / self.std
        s0 = self.stem(x3); s1 = self.l1(self.pool(s0)); s2 = self.l2(s1); s3 = self.l3(s2); s4 = self.l4(s3)
        up = lambda t, ref: F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)
        d = self.d4(torch.cat([up(s4, s3), s3], 1))
        d = self.d3(torch.cat([up(d, s2), s2], 1))
        d = self.d2(torch.cat([up(d, s1), s1], 1))
        d = self.d1(torch.cat([up(d, s0), s0], 1))
        d = self.d0(F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False))
        return self.out(d)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    print("WARNING: running on CPU — inference will be slow (minutes per image).")
model = ResNetUNet().to(DEVICE)
state = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state)                                   # half-precision tensors are converted to float32 here
model.eval()
print(f"loaded {WEIGHTS_PATH.name}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, device {DEVICE}")

In [ ]:
# ============================================================
# INFERENCE + POST-PROCESSING
# ============================================================
def load_image(path):
    img = np.array(Image.open(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    return img.astype(np.float32)

@torch.no_grad()
def predict_probability(path):
    """Flip-TTA averaged probability map at full image resolution."""
    img = load_image(path)
    h, w = img.shape
    small = cv2.resize(img, (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    views = [None, [3], [2], [2, 3]] if FLIP_TTA else [None]
    outs = []
    for dims in views:
        xi = x if dims is None else torch.flip(x, dims)
        p = torch.sigmoid(model(xi))
        if dims is not None:
            p = torch.flip(p, dims)
        outs.append(p[0, 0].float().cpu().numpy())
    return cv2.resize(np.mean(outs, axis=0), (w, h), interpolation=cv2.INTER_LINEAR)

def extract_instances(prob):
    binary = (prob > THRESHOLD).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * CLOSING_RADIUS + 1, 2 * CLOSING_RADIUS + 1))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    areas = np.bincount(labels.ravel(), minlength=n)
    return [labels == k for k in range(1, n) if areas[k] > MIN_PIXELS]

def encode_rle(mask):
    rle = mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("utf-8")

In [ ]:
# ============================================================
# RUN ON THE TEST IMAGES -> submission.csv
# ============================================================
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
test_paths = sorted(p for p in TEST_DIR.iterdir() if p.suffix.lower() in IMG_EXT)
print(len(test_paths), "test images")

rows, counts = [], []
for path in tqdm(test_paths, desc="inference"):
    instances = extract_instances(predict_probability(path))
    counts.append(len(instances))
    for k, m in enumerate(instances):
        rows.append({"filament_id": f"{path.stem}_{k}", "segmentation_rle": encode_rle(m)})

sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
sub.to_csv(OUT_CSV, index=False)

chk = pd.read_csv(OUT_CSV)
assert list(chk.columns) == ["filament_id", "segmentation_rle"] and chk.filament_id.is_unique and chk.segmentation_rle.notna().all()
assert set(chk.filament_id.str.split("_", n=1).str[0]) <= {p.stem for p in test_paths}
print(f"wrote {OUT_CSV}: {len(sub)} filaments over {sum(c > 0 for c in counts)}/{len(test_paths)} images (mean {np.mean(counts):.1f} per image)")

In [ ]:
# ============================================================
# OPTIONAL: how close is the reproduced file to the file that was submitted? (pooled PQ between the two prediction sets)
# Differences are expected to be tiny: the stored weights are half precision, which can move a few boundary pixels.
# ============================================================
def pooled_pq_between(csv_a, csv_b):
    """Treat csv_a as ground truth and csv_b as predictions (same IoU > 0.5 / many-to-many rules as the competition)."""
    A = pd.read_csv(csv_a); B = pd.read_csv(csv_b)
    A["img"] = A.filament_id.str.split("_", n=1).str[0]; B["img"] = B.filament_id.str.split("_", n=1).str[0]
    ga, gb = dict(tuple(A.groupby("img"))), dict(tuple(B.groupby("img")))
    tp_iou, fp, fn = [], 0, 0
    for img in sorted(set(ga) | set(gb)):
        ra = [{"size": [2048, 2048], "counts": c.encode()} for c in ga[img].segmentation_rle] if img in ga else []
        rb = [{"size": [2048, 2048], "counts": c.encode()} for c in gb[img].segmentation_rle] if img in gb else []
        if not ra:
            fp += len(rb); continue
        if not rb:
            fn += len(ra); continue
        iou = np.asarray(mask_util.iou(rb, ra, [0] * len(ra)))          # (n_pred, n_gt)
        hit = iou > 0.5
        tp_iou += iou[hit].tolist(); fp += int((hit.sum(1) == 0).sum()); fn += int((hit.sum(0) == 0).sum())
    den = len(tp_iou) + 0.5 * fp + 0.5 * fn
    return (sum(tp_iou) / den if den else 0.0), len(tp_iou), fp, fn

if REFERENCE_CSV is not None:
    pq, tp, fp, fn = pooled_pq_between(REFERENCE_CSV, OUT_CSV)
    print(f"agreement with {REFERENCE_CSV.name}: pooled PQ {pq:.4f} (TP {tp}, FP {fp}, FN {fn})  — 1.0 means identical predictions")
else:
    print("no reference CSV found; skipping the agreement check")

## Notes
* The competition's official scoring code is the one shared by the organizers (self-evaluation notebook); predictions are scored against every annotator record separately and pooled.
* Training code: `notebooks/05_train_final_model.ipynb`. The training notebook needs the competition data and, once, internet access to download the public torchvision ImageNet weights for the encoder initialisation.
* External resources used: torchvision ResNet-50 ImageNet weights (`IMAGENET1K_V1`, publicly available) as the starting point of the encoder.